# 89. The Single vs. Dual Sourcing Problem

## Tier 3: The Advanced Algorithm (Ant Colony Optimization)

### Key assumptions
- Sourcing problem can be modeled as a graph traversal problem
- Artificial ants can explore different sourcing combinations
- Pheromone trails represent quality of past solutions
- Heuristic information guides ants toward promising regions
- Balance between exploration and exploitation through pheromone updating

### Approach (step-by-step)
1. **Graph Construction**: Create construction graph with supplier-component nodes
2. **Ant Initialization**: Initialize colony of artificial ants with random solutions
3. **Solution Construction**: Each ant builds complete sourcing solution using probabilistic rules
4. **Pheromone Update**: Update pheromone trails based on solution quality
5. **Evaporation**: Apply pheromone evaporation to avoid local optima
6. **Convergence Monitoring**: Track best solutions and convergence behavior

### What to look for in the results
- Convergence patterns over iterations
- Pheromone trail intensity visualization
- Solution quality improvement over time
- Exploration vs. exploitation balance
- Comparison with heuristic and mathematical approaches

### Concrete example (from the source)
We'll implement ACO with the MediTech scenario:
- 5 components for more complex optimization
- 4 suppliers with varying capabilities
- 30 ants, 100 iterations, α=1.0, β=2.0, ρ=0.1
- Multi-objective fitness balancing cost and reliability

In [1]:
# Import required libraries for Ant Colony Optimization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
import random
import time
import warnings
from collections import defaultdict
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class ACOComponent:
    """Extended component data structure for ACO"""
    name: str
    criticality_score: float
    annual_demand: int
    quality_requirement: float  # Minimum quality requirement

@dataclass
class ACOSupplier:
    """Extended supplier data structure for ACO"""
    name: str
    unit_costs: Dict[str, float]
    reliability_score: float
    quality_scores: Dict[str, float]
    capacity_utilization: float
    lead_time: int
    max_capacity: int  # Maximum annual capacity

@dataclass
class ACOSolution:
    """ACO solution representation"""
    allocations: Dict[str, Dict[str, float]]  # component -> supplier -> allocation
    fitness: float
    total_cost: float
    reliability_score: float
    quality_score: float
    iteration: int
    ant_id: int

In [ ]:
class AntColonyOptimizer:
    """Ant Colony Optimization for single vs. dual sourcing problem"""
    
    def __init__(self, components: List[ACOComponent], suppliers: List[ACOSupplier],
                 n_ants: int = 30, n_iterations: int = 100,
                 alpha: float = 1.0, beta: float = 2.0, rho: float = 0.1,
                 q0: float = 0.9, cost_weight: float = 0.6, reliability_weight: float = 0.4):
        """
        Initialize ACO for sourcing optimization
        
        Args:
            components: List of components to source
            suppliers: List of available suppliers
            n_ants: Number of artificial ants in colony
            n_iterations: Number of iterations to run
            alpha: Pheromone importance parameter
            beta: Heuristic information importance parameter
            rho: Pheromone evaporation rate
            q0: Probability of choosing best-known edge (exploitation vs exploration)
            cost_weight: Weight for cost in fitness calculation
            reliability_weight: Weight for reliability in fitness calculation
        """
        self.components = components
        self.suppliers = suppliers
        self.n_ants = n_ants
        self.n_iterations = n_iterations
        self.alpha = alpha
        self.beta = beta
        self.rho = rho
        self.q0 = q0
        self.cost_weight = cost_weight
        self.reliability_weight = reliability_weight
        
        # Normalize weights
        total_weight = cost_weight + reliability_weight
        self.cost_weight /= total_weight
        self.reliability_weight /= total_weight
        
        # Initialize pheromone trails
        self.pheromone = defaultdict(lambda: defaultdict(float))
        self.initialize_pheromone()
        
        # Track best solutions
       .best_solution = None
       .best_fitness = float('inf')
       .iteration_best_solutions = []
       .convergence_history = []
        
    def initialize_pheromone(self, initial_pheromone: float = 0.1):
        """Initialize pheromone trails for all component-supplier pairs"""
        for component in self.components:
            for supplier in self.suppliers:
                self.pheromone[component.name][supplier.name] = initial_pheromone
    
    def calculate_heuristic_info(self, component: ACOComponent, supplier: ACOSupplier) -> float:
        """
        Calculate heuristic information (desirability) for component-supplier pair
        
        Higher values indicate more desirable assignments
        """
        # Cost component (inverse of cost)
        unit_cost = supplier.unit_costs[component.name]
        cost_desirability = 1.0 / (1.0 + unit_cost / 10.0)
        
        # Reliability component
        reliability_desirability = supplier.reliability_score
        
        # Quality component
        quality_desirability = supplier.quality_scores[component.name]
        
        # Capacity component (inverse of utilization)
        capacity_desirability = 1.0 - supplier.capacity_utilization
        
        # Combined heuristic information
        heuristic = (cost_desirability + reliability_desirability + 
                    quality_desirability + capacity_desirability) / 4.0
        
        return heuristic
    
    def calculate_transition_probability(self, component: ACOComponent, 
                                       supplier1: ACOSupplier, supplier2: ACOSupplier) -> float:
        """
        Calculate probability of choosing supplier1 over supplier2 for component
        """
        pheromone1 = self.pheromone[component.name][supplier1.name] ** self.alpha
        heuristic1 = self.calculate_heuristic_info(component, supplier1) ** self.beta
        
        pheromone2 = self.pheromone[component.name][supplier2.name] ** self.alpha
        heuristic2 = self.calculate_heuristic_info(component, supplier2) ** self.beta
        
        numerator = pheromone1 * heuristic1
        denominator = pheromone1 * heuristic1 + pheromone2 * heuristic2
        
        return numerator / denominator if denominator > 0 else 1.0
    
    def select_supplier_for_component(self, component: ACOComponent, 
                                       available_suppliers: List[ACOSupplier]) -> Tuple[ACOSupplier, bool]:
        """
        Select supplier for component using probabilistic decision rule
        
        Returns (selected_supplier, is_best_known_choice)
        """
        if len(available_suppliers) == 1:
            return available_suppliers[0], True
        
        # Find best known supplier based on pheromone * heuristic
        best_supplier = max(available_suppliers, 
                          key=lambda s: (self.pheromone[component.name][s.name] ** self.alpha) * 
                                       (self.calculate_heuristic_info(component, s) ** self.beta))
        
        # Decision rule: exploitation vs exploration
        if random.random() < self.q0:
            # Exploitation: choose best known
            return best_supplier, True
        else:
            # Exploration: probabilistic selection
            probabilities = []
            for supplier in available_suppliers:
                if len(available_suppliers) == 1:
                    prob = 1.0
                else:
                    other_suppliers = [s for s in available_suppliers if s != supplier]
                    prob = 0.0
                    for other_supplier in other_suppliers:
                        prob += self.calculate_transition_probability(component, supplier, other_supplier)
                    prob /= len(available_suppliers) - 1 if len(available_suppliers) > 1 else 1
                probabilities.append(prob)
            
            # Normalize probabilities
            total_prob = sum(probabilities)
            if total_prob > 0:
                probabilities = [p / total_prob for p in probabilities]
            else:
                probabilities = [1.0 / len(available_suppliers)] * len(available_suppliers)
            
            # Random selection based on probabilities
            selected_idx = np.random.choice(len(available_suppliers), p=probabilities)
            selected_supplier = available_suppliers[selected_idx]
            
            is_best = (selected_supplier == best_supplier)
            return selected_supplier, is_best
    
    def construct_solution(self, ant_id: int, iteration: int) -> ACOSolution:
        """
        Construct a complete sourcing solution for one ant
        """
        allocations = {}
        
        for component in self.components:
            # Determine if this component should use single or dual sourcing
            # High criticality components more likely to use dual sourcing
            use_dual_sourcing = (component.criticality_score > 0.7 and 
                               random.random() < 0.3)  # 30% chance for high criticality
            
            if use_dual_sourcing and len(self.suppliers) >= 2:
                # Dual sourcing: select two suppliers
                available_suppliers = self.suppliers.copy()
                random.shuffle(available_suppliers)
                
                # Select primary supplier
                primary_supplier, _ = self.select_supplier_for_component(component, available_suppliers)
                available_suppliers.remove(primary_supplier)
                
                # Select secondary supplier
                secondary_supplier, _ = self.select_supplier_for_component(component, available_suppliers)
                
                # Allocate 70-30 split (can be made adaptive)
                allocations[component.name] = {
                    primary_supplier.name: 0.7,
                    secondary_supplier.name: 0.3
                }
            else:
                # Single sourcing
                selected_supplier, _ = self.select_supplier_for_component(component, self.suppliers)
                allocations[component.name] = {selected_supplier.name: 1.0}
        
        # Calculate solution fitness
        fitness, total_cost, reliability_score, quality_score = self.calculate_solution_fitness(allocations)
        
        return ACOSolution(
            allocations=allocations,
            fitness=fitness,
            total_cost=total_cost,
            reliability_score=reliability_score,
            quality_score=quality_score,
            iteration=iteration,
            ant_id=ant_id
        )
    
    def calculate_solution_fitness(self, allocations: Dict[str, Dict[str, float]]) -> Tuple[float, float, float, float]:
        """
        Calculate multi-objective fitness for a solution
        
        Returns (fitness_score, total_cost, reliability_score, quality_score)
        Lower fitness is better (we're minimizing)
        """
        total_cost = 0
        weighted_reliability = 0
        weighted_quality = 0
        total_demand = 0
        
        for component in self.components:
            component_demand = component.annual_demand
            total_demand += component_demand
            
            for supplier_name, allocation in allocations[component.name].items():
                supplier = next(s for s in self.suppliers if s.name == supplier_name)
                
                # Cost contribution
                unit_cost = supplier.unit_costs[component.name]
                total_cost += component_demand * allocation * unit_cost
                
                # Reliability contribution
                weighted_reliability += component_demand * allocation * supplier.reliability_score
                
                # Quality contribution
                quality_score = supplier.quality_scores[component.name]
                weighted_quality += component_demand * allocation * quality_score
        
        # Normalize scores
        avg_reliability = weighted_reliability / total_demand if total_demand > 0 else 0
        avg_quality = weighted_quality / total_demand if total_demand > 0 else 0
        
        # Multi-objective fitness (lower is better)
        normalized_cost = total_cost / 10000  # Normalize by typical cost scale
        normalized_reliability = 1 - avg_reliability  # Convert to minimization
        normalized_quality = 1 - avg_quality  # Convert to minimization
        
        fitness = (self.cost_weight * normalized_cost + 
                  self.reliability_weight * normalized_reliability +
                  0.1 * normalized_quality)  # Small weight for quality
        
        return fitness, total_cost, avg_reliability, avg_quality
    
    def update_pheromones(self, solutions: List[ACOSolution]):
        """
        Update pheromone trails based on solution quality
        """
        # Evaporation
        for component in self.components:
            for supplier in self.suppliers:
                self.pheromone[component.name][supplier.name] *= (1 - self.rho)
        
        # Deposit pheromone based on solution quality
        for solution in solutions:
            pheromone_deposit = 1.0 / (1.0 + solution.fitness)  # Better solutions deposit more
            
            for component_name, allocations in solution.allocations.items():
                for supplier_name, allocation in allocations.items():
                    # Deposit proportional to allocation and solution quality
                    deposit_amount = pheromone_deposit * allocation
                    self.pheromone[component_name][supplier_name] += deposit_amount
    
    def optimize(self) -> ACOSolution:
        """
        Run the complete ACO optimization process
        """
        print(f"Starting ACO optimization with {self.n_ants} ants and {self.n_iterations} iterations")
        print(f"Parameters: α={self.alpha}, β={self.beta}, ρ={self.rho}, q₀={self.q0}")
        
        start_time = time.time()
        
        for iteration in range(self.n_iterations):
            iteration_solutions = []
            iteration_best_solution = None
            iteration_best_fitness = float('inf')
            
            # Each ant constructs a solution
            for ant_id in range(self.n_ants):
                solution = self.construct_solution(ant_id, iteration)
                iteration_solutions.append(solution)
                
                # Track iteration best
                if solution.fitness < iteration_best_fitness:
                    iteration_best_fitness = solution.fitness
                    iteration_best_solution = solution
                
                # Track global best
                if solution.fitness < self.best_fitness:
                    self.best_fitness = solution.fitness
                    self.best_solution = solution
            
            # Update pheromones
            self.update_pheromones(iteration_solutions)
            
            # Record convergence data
            self.convergence_history.append({
                'iteration': iteration + 1,
                'best_fitness': self.best_fitness,
                'iteration_best_fitness': iteration_best_fitness,
                'avg_fitness': np.mean([s.fitness for s in iteration_solutions]),
                'worst_fitness': max([s.fitness for s in iteration_solutions])
            })
            
            self.iteration_best_solutions.append(iteration_best_solution)
            
            # Progress reporting
            if (iteration + 1) % 20 == 0:
                print(f"Iteration {iteration + 1}: Best Fitness = {self.best_fitness:.4f}")
        
        total_time = time.time() - start_time
        print(f"\nACO optimization completed in {total_time:.2f} seconds")
        print(f"Best fitness achieved: {self.best_fitness:.4f}")
        
        return self.best_solution

In [ ]:
# Create the extended concrete example for ACO
def create_aco_example():
    """Create extended example with 5 components and 4 suppliers"""
    
    # 5 components with varying criticality
    components = [
        ACOComponent(
            name="Component_1_Critical",
            criticality_score=0.95,
            annual_demand=460,
            quality_requirement=0.90
        ),
        ACOComponent(
            name="Component_2_Standard",
            criticality_score=0.65,
            annual_demand=810,
            quality_requirement=0.85
        ),
        ACOComponent(
            name="Component_3_Medium",
            criticality_score=0.75,
            annual_demand=320,
            quality_requirement=0.88
        ),
        ACOComponent(
            name="Component_4_Low",
            criticality_score=0.45,
            annual_demand=550,
            quality_requirement=0.80
        ),
        ACOComponent(
            name="Component_5_High",
            criticality_score=0.85,
            annual_demand=280,
            quality_requirement=0.92
        )
    ]
    
    # 4 suppliers with diverse capabilities
    suppliers = [
        ACOSupplier(
            name="Supplier_A",
            unit_costs={
                "Component_1_Critical": 15,
                "Component_2_Standard": 8,
                "Component_3_Medium": 12,
                "Component_4_Low": 6,
                "Component_5_High": 18
            },
            reliability_score=0.85,
            quality_scores={
                "Component_1_Critical": 0.87,
                "Component_2_Standard": 0.86,
                "Component_3_Medium": 0.88,
                "Component_4_Low": 0.84,
                "Component_5_High": 0.89
            },
            capacity_utilization=0.75,
            lead_time=14,
            max_capacity=2000
        ),
        ACOSupplier(
            name="Supplier_B",
            unit_costs={
                "Component_1_Critical": 18,
                "Component_2_Standard": 7,
                "Component_3_Medium": 14,
                "Component_4_Low": 8,
                "Component_5_High": 20
            },
            reliability_score=0.92,
            quality_scores={
                "Component_1_Critical": 0.94,
                "Component_2_Standard": 0.91,
                "Component_3_Medium": 0.93,
                "Component_4_Low": 0.90,
                "Component_5_High": 0.95
            },
            capacity_utilization=0.65,
            lead_time=10,
            max_capacity=2500
        ),
        ACOSupplier(
            name="Supplier_C",
            unit_costs={
                "Component_1_Critical": 16,
                "Component_2_Standard": 9,
                "Component_3_Medium": 13,
                "Component_4_Low": 7,
                "Component_5_High": 19
            },
            reliability_score=0.88,
            quality_scores={
                "Component_1_Critical": 0.90,
                "Component_2_Standard": 0.89,
                "Component_3_Medium": 0.91,
                "Component_4_Low": 0.87,
                "Component_5_High": 0.92
            },
            capacity_utilization=0.80,
            lead_time=12,
            max_capacity=1800
        ),
        ACOSupplier(
            name="Supplier_D",
            unit_costs={
                "Component_1_Critical": 14,
                "Component_2_Standard": 10,
                "Component_3_Medium": 11,
                "Component_4_Low": 9,
                "Component_5_High": 17
            },
            reliability_score=0.82,
            quality_scores={
                "Component_1_Critical": 0.85,
                "Component_2_Standard": 0.83,
                "Component_3_Medium": 0.86,
                "Component_4_Low": 0.81,
                "Component_5_High": 0.87
            },
            capacity_utilization=0.70,
            lead_time=16,
            max_capacity=2200
        )
    ]
    
    return components, suppliers

# Create the instance
components, suppliers = create_aco_example()
print(f"Created {len(components)} components and {len(suppliers)} suppliers")
print(f"Components: {[c.name for c in components]}")
print(f"Suppliers: {[s.name for s in suppliers]}")

In [ ]:
# Initialize and run the ACO optimizer
aco = AntColonyOptimizer(
    components=components,
    suppliers=suppliers,
    n_ants=30,
    n_iterations=100,
    alpha=1.0,  # Pheromone importance
    beta=2.0,   # Heuristic importance
    rho=0.1,    # Evaporation rate
    q0=0.9,     # Exploitation probability
    cost_weight=0.6,
    reliability_weight=0.4
)

# Run optimization
best_solution = aco.optimize()

print("\n=== BEST ACO SOLUTION ===")
print(f"Fitness Score: {best_solution.fitness:.4f}")
print(f"Total Annual Cost: ${best_solution.total_cost:,.2f}")
print(f"Reliability Score: {best_solution.reliability_score:.3f}")
print(f"Quality Score: {best_solution.quality_score:.3f}")

print("\n=== OPTIMAL SOURCING STRATEGY ===")
for component_name, allocations in best_solution.allocations.items():
    print(f"\n{component_name}:")
    if len(allocations) == 1:
        supplier_name = list(allocations.keys())[0]
        print(f"  Single Sourcing: 100% from {supplier_name}")
    else:
        print(f"  Dual Sourcing:")
        for supplier_name, allocation in allocations.items():
            print(f"    {allocation*100:.0f}% from {supplier_name}")

In [ ]:
# Analyze convergence and solution quality
def analyze_convergence():
    """Analyze ACO convergence behavior"""
    
    convergence_df = pd.DataFrame(aco.convergence_history)
    
    print("=== CONVERGENCE ANALYSIS ===")
    
    # Find convergence point (when improvement becomes small)
    improvements = []
    for i in range(1, len(convergence_df)):
        improvement = (convergence_df.iloc[i-1]['best_fitness'] - 
                      convergence_df.iloc[i]['best_fitness'])
        improvements.append(improvement)
    
    # Convergence threshold: when improvement < 0.1% of current best
    convergence_iteration = None
    for i, improvement in enumerate(improvements):
        if improvement < 0.001 * convergence_df.iloc[i]['best_fitness']:
            convergence_iteration = i + 1
            break
    
    if convergence_iteration:
        print(f"Convergence achieved at iteration {convergence_iteration}")
        print(f"Final fitness: {convergence_df.iloc[-1]['best_fitness']:.4f}")
    else:
        print(f"No clear convergence within {len(convergence_df)} iterations")
    
    # Solution diversity analysis
    final_solutions = aco.iteration_best_solutions[-10:]  # Last 10 iterations
    unique_strategies = set()
    
    for solution in final_solutions:
        strategy_signature = ""
        for component_name in sorted(solution.allocations.keys()):
            allocations = solution.allocations[component_name]
            if len(allocations) == 1:
                strategy_signature += "S"  # Single
            else:
                strategy_signature += "D"  # Dual
        unique_strategies.add(strategy_signature)
    
    print(f"Solution diversity: {len(unique_strategies)} unique strategies in final 10 iterations")
    
    return convergence_df

convergence_df = analyze_convergence()

In [ ]:
# Compare with heuristic and mathematical approaches
def compare_with_baselines():
    """Compare ACO results with baseline methods"""
    
    print("=== BASELINE COMPARISON ===")
    
    # Simple baseline: cheapest supplier for each component (single sourcing)
    cheapest_cost = 0
    cheapest_allocations = {}
    
    for component in components:
        cheapest_supplier = min(suppliers, key=lambda s: s.unit_costs[component.name])
        component_cost = component.annual_demand * cheapest_supplier.unit_costs[component.name]
        cheapest_cost += component_cost
        cheapest_allocations[component.name] = {cheapest_supplier.name: 1.0}
    
    # Calculate fitness for cheapest solution
    cheapest_fitness, _, cheapest_reliability, _ = aco.calculate_solution_fitness(cheapest_allocations)
    
    # Most reliable baseline: most reliable supplier for each component
    most_reliable_cost = 0
    most_reliable_allocations = {}
    
    for component in components:
        most_reliable_supplier = max(suppliers, key=lambda s: s.reliability_score)
        component_cost = component.annual_demand * most_reliable_supplier.unit_costs[component.name]
        most_reliable_cost += component_cost
        most_reliable_allocations[component.name] = {most_reliable_supplier.name: 1.0}
    
    # Calculate fitness for most reliable solution
    most_reliable_fitness, _, most_reliable_reliability, _ = aco.calculate_solution_fitness(most_reliable_allocations)
    
    # Create comparison table
    comparison_data = [
        {
            'Method': 'ACO (Best)',
            'Total_Cost': best_solution.total_cost,
            'Reliability': best_solution.reliability_score,
            'Fitness': best_solution.fitness,
            'Dual_Sourcing': sum(1 for allocs in best_solution.allocations.values() if len(allocs) > 1)
        },
        {
            'Method': 'Cheapest (Single)',
            'Total_Cost': cheapest_cost,
            'Reliability': cheapest_reliability,
            'Fitness': cheapest_fitness,
            'Dual_Sourcing': 0
        },
        {
            'Method': 'Most Reliable (Single)',
            'Total_Cost': most_reliable_cost,
            'Reliability': most_reliable_reliability,
            'Fitness': most_reliable_fitness,
            'Dual_Sourcing': 0
        }
    ]
    
    comparison_df = pd.DataFrame(comparison_data)
    print(comparison_df.to_string(index=False))
    
    # Calculate improvements
    cost_improvement_vs_cheapest = ((cheapest_cost - best_solution.total_cost) / cheapest_cost) * 100
    reliability_improvement_vs_cheapest = ((best_solution.reliability_score - cheapest_reliability) / cheapest_reliability) * 100
    
    print(f"\nACO vs. Cheapest Single Sourcing:")
    print(f"  Cost difference: {cost_improvement_vs_cheapest:+.1f}%")
    print(f"  Reliability improvement: {reliability_improvement_vs_cheapest:+.1f}%")
    
    return comparison_df

comparison_df = compare_with_baselines()

In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Ant Colony Optimization Analysis', fontsize=16, fontweight='bold')

# 1. Convergence Plot
ax1 = axes[0, 0]
ax1.plot(convergence_df['iteration'], convergence_df['best_fitness'], 
         'b-', linewidth=2, label='Best Fitness')
ax1.plot(convergence_df['iteration'], convergence_df['iteration_best_fitness'], 
         'g--', alpha=0.7, label='Iteration Best')
ax1.plot(convergence_df['iteration'], convergence_df['avg_fitness'], 
         'r:', alpha=0.5, label='Average Fitness')
ax1.fill_between(convergence_df['iteration'], 
                 convergence_df['best_fitness'], convergence_df['worst_fitness'],
                 alpha=0.2, color='gray', label='Fitness Range')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Fitness Score (lower is better)')
ax1.set_title('ACO Convergence Behavior')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Pheromone Trail Heatmap
ax2 = axes[0, 1]
pheromone_matrix = []
component_names = [c.name.replace('_', ' ').title() for c in components]
supplier_names = [s.name.replace('Supplier_', 'S') for s in suppliers]

for component in components:
    row = [aco.pheromone[component.name][supplier.name] for supplier in suppliers]
    pheromone_matrix.append(row)

im = ax2.imshow(pheromone_matrix, cmap='YlOrRd', aspect='auto')
ax2.set_xticks(range(len(supplier_names)))
ax2.set_yticks(range(len(component_names)))
ax2.set_xticklabels(supplier_names)
ax2.set_yticklabels(component_names)
ax2.set_title('Final Pheromone Trail Intensity')

# Add pheromone values as text
for i in range(len(component_names)):
    for j in range(len(supplier_names)):
        text = ax2.text(j, i, f'{pheromone_matrix[i][j]:.2f}',
                       ha="center", va="center", color="black", fontweight='bold')

plt.colorbar(im, ax=ax2, label='Pheromone Intensity')

# 3. Method Comparison
ax3 = axes[1, 0]
x_pos = np.arange(len(comparison_df))
width = 0.25

# Cost comparison
bars1 = ax3.bar(x_pos - width, comparison_df['Total_Cost'], width, 
                label='Total Cost', color='lightcoral', alpha=0.8)

# Reliability comparison (scaled for visibility)
scaled_reliability = comparison_df['Reliability'] * 10000  # Scale to cost range
bars2 = ax3.bar(x_pos, scaled_reliability, width, 
                label='Reliability (×10000)', color='lightblue', alpha=0.8)

# Dual sourcing count
bars3 = ax3.bar(x_pos + width, comparison_df['Dual_Sourcing'] * 1000, width,
                label='Dual Sourcing (×1000)', color='lightgreen', alpha=0.8)

ax3.set_xlabel('Method')
ax3.set_ylabel('Value')
ax3.set_title('Multi-Criteria Comparison')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(comparison_df['Method'], rotation=45)
ax3.legend()

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax3.text(bar.get_x() + bar.get_width()/2., height + 50,
                    f'{height:.0f}', ha='center', va='bottom', fontsize=8)

# 4. Solution Strategy Distribution
ax4 = axes[1, 1]
strategy_counts = {'Single Sourcing': 0, 'Dual Sourcing': 0}

for allocations in best_solution.allocations.values():
    if len(allocations) == 1:
        strategy_counts['Single Sourcing'] += 1
    else:
        strategy_counts['Dual Sourcing'] += 1

colors = ['lightcoral', 'lightblue']
wedges, texts, autotexts = ax4.pie(strategy_counts.values(), 
                                  labels=[f'{k} ({v})' for k, v in strategy_counts.items()],
                                  autopct='%1.1f%%', colors=colors, startangle=90)
ax4.set_title('ACO Optimal Strategy Distribution')

# Make percentage text bold
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

plt.tight_layout()
plt.show()

In [ ]:
# Detailed pheromone analysis and solution insights
def analyze_pheromone_patterns():
    """Analyze pheromone patterns for insights"""
    
    print("=== PHEROMONE ANALYSIS ===")
    
    for component in components:
        print(f"\n{component.name} (Criticality: {component.criticality_score:.2f}):")
        
        # Get pheromone levels for this component
        supplier_pheromones = [(s.name, aco.pheromone[component.name][s.name]) 
                               for s in suppliers]
        supplier_pheromones.sort(key=lambda x: x[1], reverse=True)
        
        print("  Pheromone Levels:")
        for i, (supplier_name, pheromone_level) in enumerate(supplier_pheromones, 1):
            supplier = next(s for s in suppliers if s.name == supplier_name)
            print(f"    {i}. {supplier_name}: {pheromone_level:.3f}")
            print(f"       Cost: ${supplier.unit_costs[component.name]}, Reliability: {supplier.reliability_score:.2f}")
        
        # Show final decision
        allocations = best_solution.allocations[component.name]
        if len(allocations) == 1:
            supplier_name = list(allocations.keys())[0]
            print(f"  Final Decision: Single sourcing from {supplier_name}")
        else:
            print(f"  Final Decision: Dual sourcing")
            for supplier_name, allocation in allocations.items():
                print(f"    {allocation*100:.0f}% from {supplier_name}")
    
    # Pheromone distribution analysis
    print("\n=== PHEROMONE DISTRIBUTION ANALYSIS ===")
    all_pheromones = []
    for component in components:
        for supplier in suppliers:
            all_pheromones.append(aco.pheromone[component.name][supplier.name])
    
    print(f"Mean pheromone level: {np.mean(all_pheromones):.3f}")
    print(f"Std deviation: {np.std(all_pheromones):.3f}")
    print(f"Min pheromone: {np.min(all_pheromones):.3f}")
    print(f"Max pheromone: {np.max(all_pheromones):.3f}")
    print(f"Pheromone range: {np.max(all_pheromones) / np.min(all_pheromones):.1f}x")

analyze_pheromone_patterns()

### Why this Tier exists vs earlier Tiers
This Tier 3 provides advanced metaheuristic optimization that addresses limitations of simpler methods:
- **Global search capability**: Explores diverse solution regions to avoid local optima
- **Adaptive learning**: Pheromone trails capture collective intelligence over time
- **Balance exploration/exploitation**: Systematically manages search diversity vs. convergence
- **Population-based search**: Multiple parallel solutions provide robustness
- **Emergent optimization**: Complex patterns emerge from simple ant behaviors

### Pros / Cons vs earlier Tiers
**Advantages over Tier 1 (Mathematical Optimization):**
- **Scalability**: Handles much larger problem instances efficiently
- **Flexibility**: Easily incorporates complex constraints and objectives
- **Robustness**: Less sensitive to parameter variations and data quality
- **Near-optimal solutions**: Often finds solutions within 1-5% of optimal
- **Parallelizable**: Natural parallel computation across ant colony

**Advantages over Tier 2 (Priority Heuristic):**
- **Global search**: Explores multiple solution regions vs. greedy single-path
- **Learning capability**: Improves over time through pheromone accumulation
- **Solution diversity**: Maintains population of diverse solutions
- **Adaptive behavior**: Responds to problem structure and landscape
- **Better quality**: Typically finds superior solutions to simple heuristics

**Disadvantages:**
- **Parameter tuning**: Performance sensitive to ACO parameter settings
- **Computational cost**: More expensive than simple heuristics
- **Convergence uncertainty**: No guarantee of finding global optimum
- **Complexity**: More complex to implement and debug

### When to use this Tier
- **Medium to large problems**: When heuristics are insufficient but exact methods are too slow
- **Complex constraints**: When problem has many interacting constraints
- **Multi-objective optimization**: When balancing conflicting objectives is critical
- **Robust solutions needed**: When solution quality consistency is important
- **Limited time for exact methods**: When good-enough solutions quickly are valuable
- **Learning from past solutions**: When historical solution patterns can be leveraged